# Tugas Mandiri: Implementasi Fase-Fase Kompilator
**Mata Kuliah:** Teknik Kompilasi
**Topik:** Lexer, Parser (EBNF), AST, dan TAC

---
## Deskripsi Tugas
Berdasarkan materi yang telah dipelajari mengenai *Lexical Analysis*, *EBNF*, dan *Abstract Syntax Tree*, Anda diminta untuk melengkapi sebuah **Mini Compiler**.

### Pengumpulan Tugas
- Tugas dikerjakan dalam repositori github masing-masing. Pastikan repositori di-set public.
- link github dikumpulkan di halaman UTS web mentari/ e-learning
- Maksimal pengumpulan tugas 13 Mei 2026

**Tantangan Utama:** Tambahkan dukungan untuk operator pangkat (`^`). Dalam hirarki matematika, pangkat memiliki prioritas lebih tinggi daripada perkalian (`*`) dan pembagian (`/`).

### 1. Definisi Node AST
Bagian ini mendefinisikan struktur data pohon untuk representasi kode.

In [2]:
class AST:
    pass

class BinOp(AST):
    def __init__(self, left, op, right):
        self.left = left
        self.op = op
        self.right = right

class Num(AST):
    def __init__(self, value):
        self.value = value

class Var(AST):
    def __init__(self, name):
        self.name = name

class ParserError(Exception):
    pass

### 2. Implementasi MiniCompiler
Lengkapi bagian bertanda `TUGAS` di bawah ini.

In [8]:
import re

# =========================
# AST NODE DEFINITIONS
# =========================

class AST:
    pass

class BinOp(AST):
    def __init__(self, left, op, right):
        self.left = left
        self.op = op
        self.right = right

class Num(AST):
    def __init__(self, value):
        self.value = value

class Var(AST):
    def __init__(self, name):
        self.name = name

class ParserError(Exception):
    pass


# =========================
# MINI COMPILER
# =========================

class MiniCompiler:
    def __init__(self, source, env):

        # SUPPORT OPERATOR ^
        self._tokens = iter(
            re.findall(
                r'[a-zA-Z_]\w*|\d+(?:\.\d+)?|[\^+*/()\-]',
                source
            ) + ['?']
        )

        self._current = None
        self._env = env
        self._temp_count = 0

        self.advance()

    def advance(self):
        try:
            self._current = next(self._tokens)
        except StopIteration:
            self._current = None

    def expect(self, expected):
        if self._current != expected and not (
            expected == "ID" and self._current.isalnum()
        ):
            raise ParserError(
                f"Expected {expected}, found {self._current}"
            )

        token = self._current
        self.advance()
        return token

    # =========================
    # FACTOR
    # =========================
    def factor(self):

        token = self._current

        if token is not None and token.replace('.', '', 1).isdigit():
            self.advance()
            return Num(float(token) if '.' in token else int(token))

        elif token and token.isalpha():

            if token not in self._env:
                raise ParserError(
                    f"Semantic Error: Undefined variable '{token}'"
                )

            self.advance()
            return Var(token)

        elif token == '(':

            self.advance()
            node = self.expr()
            self.expect(')')
            return node

        raise ParserError(f"Unexpected token: {token}")

    # =========================
    # POWER (^)
    # =========================
    def power(self):

        node = self.factor()

        while self._current == '^':

            op = self._current
            self.advance()

            node = BinOp(
                left=node,
                op=op,
                right=self.factor()
            )

        return node

    # =========================
    # TERM (* /)
    # =========================
    def term(self):

        node = self.power()

        while self._current in ('*', '/'):

            op = self._current
            self.advance()

            node = BinOp(
                left=node,
                op=op,
                right=self.power()
            )

        return node

    # =========================
    # EXPRESSION (+ -)
    # =========================
    def expr(self):

        node = self.term()

        while self._current in ('+', '-'):

            op = self._current
            self.advance()

            node = BinOp(
                left=node,
                op=op,
                right=self.term()
            )

        return node

    # =========================
    # TAC GENERATOR
    # =========================
    def generate_tac(self, node):

        if isinstance(node, Num):
            return str(node.value)

        if isinstance(node, Var):
            return node.name

        left_val = self.generate_tac(node.left)
        right_val = self.generate_tac(node.right)

        self._temp_count += 1
        temp_name = f"t{self._temp_count}"

        print(f"{temp_name} = {left_val} {node.op} {right_val}")

        return temp_name


# =========================
# TESTING
# =========================

source_code = "a ^ 2 + b * c"
symbol_table = {
    'a': 5,
    'b': 10,
    'c': 2
}

try:

    print(f"Input: {source_code}")

    compiler = MiniCompiler(
        source_code,
        symbol_table
    )

    ast_root = compiler.expr()

    print("\n--- Output Three Address Code (TAC) ---")

    compiler.generate_tac(ast_root)

except Exception as e:
    print(f"Error: {e}")

Input: a ^ 2 + b * c

--- Output Three Address Code (TAC) ---
t1 = a ^ 2
t2 = b * c
t3 = t1 + t2


### 3. Uji Coba
Gunakan sel ini untuk menguji implementasi Anda.

In [9]:
source_code = "a ^ 2 + b * c"
symbol_table = {'a': 5, 'b': 10, 'c': 2}

try:
    print(f"Input: {source_code}")
    compiler = MiniCompiler(source_code, symbol_table)
    ast_root = compiler.expr()

    print("\n--- Output Three Address Code (TAC) ---")
    compiler.generate_tac(ast_root)
except Exception as e:
    print(f"Error: {e}")

Input: a ^ 2 + b * c

--- Output Three Address Code (TAC) ---
t1 = a ^ 2
t2 = b * c
t3 = t1 + t2



1. Mengapa fungsi `power()` harus dipanggil di dalam `term()`, bukan sebaliknya? Jelaskan kaitannya dengan *Operator Precedence*.
Fungsi power() dipanggil di dalam term() karena operator ^ memiliki prioritas (precedence) lebih tinggi dibanding * atau /. Jadi operasi pangkat harus dihitung lebih dulu sebelum perkalian atau pembagian agar hasil ekspresi benar.
2. Apa yang terjadi pada fase **Analisis Semantik** jika variabel `z` digunakan dalam kode sumber tetapi tidak ada di `symbol_table`?
Pada fase Analisis Semantik, jika variabel z digunakan tetapi tidak ada di symbol_table, maka compiler akan menghasilkan semantic error karena variabel dianggap belum dideklarasikan atau tidak dikenali.
3. Jelaskan mengapa dalam TAC, instruksi untuk `a ^ 2` harus muncul sebelum instruksi untuk `+`.
Dalam TAC (Three Address Code), instruksi a ^ 2 harus muncul sebelum + karena operasi pangkat harus diselesaikan terlebih dahulu sesuai aturan prioritas operator. Hasil pangkat disimpan sementara, lalu baru digunakan pada operasi penjumlahan.